# mesh_genai — Quad Training on Colab

**Phase 5: Real training run with Objaverse data, Wandb logging, and Drive persistence.**

Runtime requirements:
- Runtime → Change runtime type → **GPU** (T4 recommended)
- Expected time for 500-step validation: ~30–45 min
- Expected time for full 50 000-step run: ~8–12 h (across multiple sessions)

Session reconnect: checkpoints are synced to Drive every 1 000 steps.  
To resume: run all cells and set `RESUME_FROM_CKPT` in Cell 6.

## Cell 1 — GPU check + Drive mount

In [ ]:
import subprocess, sys, os, time

# ── GPU check ────────────────────────────────────────────────────────────────
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found.  Go to Runtime → Change runtime type → GPU.')
gpu_name, gpu_mem = result.stdout.strip().split(',')
print(f'GPU: {gpu_name.strip()}  |  VRAM: {gpu_mem.strip()}')

# Recommend batch size based on VRAM
vram_mib = int(gpu_mem.strip().split()[0])
suggested_batch = 64 if vram_mib >= 20000 else (32 if vram_mib >= 14000 else 16)
print(f'Suggested batch_size for 12-token faces: {suggested_batch}')

# ── Google Drive mount ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

## Cell 2 — Clone repo and install dependencies

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GITHUB_REPO  = 'https://github.com/vixrino/meshXYZ.git'
BRANCH       = 'main'
REPO_DIR     = '/content/meshXYZ'

# ── Clone ────────────────────────────────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── Python dependencies ───────────────────────────────────────────────────────
# Core ML stack (Colab already has torch 2.x; install extras)
!pip install -q \
    lightning==2.2.* \
    wandb \
    dacite \
    jaxtyping \
    einops \
    trimesh \
    objaverse

# torch-geometric and torch-cluster (version pinned to current torch)
import torch
TORCH_VER = torch.__version__.split('+')[0]          # e.g. '2.2.1'
CUDA_TAG  = 'cu121' if torch.cuda.is_available() else 'cpu'
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {WHEEL_URL}

print('All dependencies installed.')

## Cell 3 — Encoder weights

In [ ]:
# Download pretrained KLAutoEncoder weights.
# Source: https://drive.google.com/file/d/1qX_YTMAE2tLFppJps3vKAWFbFZgE9CtQ
#
# Option A — gdown (works for public Drive links):
!pip install -q gdown
ENCODER_FILE_ID  = '1qX_YTMAE2tLFppJps3vKAWFbFZgE9CtQ'
ENCODER_LOCAL    = '/content/encoder.pth'

if not os.path.exists(ENCODER_LOCAL):
    !gdown {ENCODER_FILE_ID} -O {ENCODER_LOCAL}
    print(f'Encoder saved to {ENCODER_LOCAL}  ({os.path.getsize(ENCODER_LOCAL)/1e6:.1f} MB)')
else:
    print(f'Encoder already cached at {ENCODER_LOCAL}')

# Sanity-check: load weights
import torch
ckpt = torch.load(ENCODER_LOCAL, map_location='cpu', weights_only=False)
print('Encoder keys:', list(ckpt.keys())[:5], '...')

## Cell 4 — Dataset preparation (Objaverse)

In [ ]:
# ── Dataset configuration ─────────────────────────────────────────────────────
N_TRAIN    = 500     # start small to validate; scale to 5000 once confirmed healthy
N_VAL      = 50
TRAIN_DIR  = '/content/data/train'
VAL_DIR    = '/content/data/val'

# Check if data was already downloaded in a previous session (restored from Drive)
DRIVE_DATA = '/content/drive/MyDrive/mesh_genai/data'

def count_meshes(directory):
    import glob
    total = 0
    for ext in ('*.obj', '*.ply', '*.glb', '*.gltf'):
        total += len(glob.glob(os.path.join(directory, '**', ext), recursive=True))
    return total

# Restore from Drive cache if present
for split, local_dir, n in [('train', TRAIN_DIR, N_TRAIN), ('val', VAL_DIR, N_VAL)]:
    drive_cached = os.path.join(DRIVE_DATA, split)
    if os.path.isdir(drive_cached) and count_meshes(drive_cached) >= n * 0.8:
        print(f'Restoring {split} data from Drive cache ({count_meshes(drive_cached)} meshes)')
        import shutil
        if not os.path.isdir(local_dir):
            shutil.copytree(drive_cached, local_dir)
    else:
        print(f'Downloading {split} split ({n} objects) …')
        os.makedirs(local_dir, exist_ok=True)
        !python scripts/prep_objaverse.py \
            --n {n} \
            --out_dir {local_dir} \
            --split {split} \
            --processes 4

        # Save to Drive for future sessions
        os.makedirs(DRIVE_DATA, exist_ok=True)
        print(f'Backing up {split} data to Drive …')
        import shutil
        drive_dst = os.path.join(DRIVE_DATA, split)
        if os.path.isdir(drive_dst):
            shutil.rmtree(drive_dst)
        shutil.copytree(local_dir, drive_dst)

print(f'Train: {count_meshes(TRAIN_DIR)} meshes')
print(f'Val:   {count_meshes(VAL_DIR)} meshes')

In [ ]:
# Quick topology audit — shows how many meshes have native quads
import csv, glob
manifest = os.path.join(TRAIN_DIR, 'manifest.csv')
if os.path.exists(manifest):
    with open(manifest) as f:
        rows = list(csv.DictReader(f))
    kept  = [r for r in rows if r['kept'] == 'True']
    quads = [r for r in kept if float(r['quad_frac']) > 0.05]
    print(f'Kept: {len(kept)}  |  Quad-rich (>5% quads): {len(quads)} ({len(quads)/max(len(kept),1):.0%})')
    fmts  = {}
    for r in kept:
        fmts[r['format']] = fmts.get(r['format'], 0) + 1
    print('Formats:', fmts)
else:
    print('No manifest found — topology audit skipped.')

## Cell 5 — Wandb login

In [ ]:
import wandb
# Paste your API key or set WANDB_API_KEY env var before running.
# wandb.login(key='YOUR_API_KEY')  # ← uncomment and paste key, OR:
wandb.login()   # interactive prompt

# Verify login
print('Wandb entity:', wandb.api.default_entity)

## Cell 6 — Training configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CONFIG_PATH   = 'config/config-quad.yaml'
OUTPUT_DIR    = '/content/runs/quad'
RESUME_FROM_CKPT = None   # ← set to Drive ckpt path to resume, e.g.:
                           # '/content/drive/MyDrive/mesh_genai/runs/run_20260613_210000/last.ckpt'

# ── Override encoder weights path in config ──────────────────────────────────
# We patch the yaml at runtime rather than editing it on disk.
import yaml
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg['encoder']['weights_path'] = ENCODER_LOCAL
cfg['training']['batch_size']  = suggested_batch   # set from GPU check above

patched_cfg = '/tmp/config-quad-patched.yaml'
with open(patched_cfg, 'w') as f:
    yaml.dump(cfg, f)

print(f'Encoder weights : {cfg["encoder"]["weights_path"]}')
print(f'Batch size      : {cfg["training"]["batch_size"]}')
print(f'Resume from     : {RESUME_FROM_CKPT or "scratch"}')

## Cell 7 — Launch training

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [
    'python', '-m', 'src.train',
    '--train_dir',  TRAIN_DIR,
    '--val_dir',    VAL_DIR,
    '--config',     patched_cfg,
    '--output_dir', OUTPUT_DIR,
]
if RESUME_FROM_CKPT:
    cmd += ['--ckpt', RESUME_FROM_CKPT]

print('Launching training …')
print(' '.join(cmd))
print()

# Run training (blocks until done or interrupted)
import subprocess
result = subprocess.run(cmd, cwd=REPO_DIR)
print('Exit code:', result.returncode)

## Cell 8 — Sanity check after first 500 steps

Run this cell while training is ongoing (or pause and inspect a checkpoint) to verify the run is healthy.

**Expected numbers at step 500 (N=500 meshes, batch_size=32, d_hidden=384):**

| Metric | Healthy range | Red flag |
|--------|--------------|----------|
| `train/loss` | 3.5 – 4.5 | > 5.4 (not learning) |
| `train/loss_tri_pad` | 2.0 – 3.5 | > 5.0 (TRI_PAD not learning) |
| `train/loss_coord` | 3.5 – 4.5 | same as init (~5.5) |
| `train/loss_eos` | 1.0 – 3.0 | > 5.0 (no EOS signal) |
| `train/face_type_acc` | 0.30 – 0.65 | < 0.15 (mode collapse) |
| `train/eos_acc` | > 0.85 | < 0.50 |
| `train/coord_acc` | 0.005 – 0.03 | > 0.10 (overfitting) |
| grad norm (pre-clip) | 0.5 – 3.0 | > 10 (explosion) |

**Loss curve shape:**
- `loss_tri_pad` should drop faster than `loss_coord` — it's a deterministic signal.
- `loss_eos` should be very low early (rare token, easy to learn).
- `face_type_acc` will show the three-phase pattern (see Phase 4 smoke test):
  expect the mode-switch valley around step 200–400 on diverse data.

In [ ]:
# Pull the last 500 rows from wandb and plot
import wandb, pandas as pd, matplotlib.pyplot as plt

api = wandb.Api()
PROJECT = cfg['wandb']['project']  # 'mesh_genai'

# Get most recent run
runs = api.runs(f"{wandb.api.default_entity}/{PROJECT}", order="-created_at")
run  = runs[0]
print(f'Analysing run: {run.name}  (id={run.id})  state={run.state}')

hist = run.history(samples=1000, keys=[
    'train/loss', 'train/loss_tri_pad', 'train/loss_coord',
    'train/loss_eos', 'train/face_type_acc', 'train/eos_acc', 'train/coord_acc',
], pandas=True)

if hist.empty:
    print('No history yet — run more steps first.')
else:
    # Print last-row summary
    last = hist.iloc[-1]
    print(f'\nAt step {int(last["_step"])}:')
    for col in hist.columns:
        if col.startswith('train/'):
            val = last[col]
            print(f'  {col:<30s} {val:.4f}')

    # ── Plot ─────────────────────────────────────────────────────────────────
    def smooth(s, a=0.3):
        out = [s.iloc[0]]
        for v in s.iloc[1:]:
            out.append(a*v + (1-a)*out[-1])
        return pd.Series(out, index=s.index)

    steps = hist['_step']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Run: {run.name}  — sanity check @ step {int(last["_step"])}')

    # Loss components
    ax = axes[0]
    for col, color in [('train/loss','#2c3e50'), ('train/loss_tri_pad','#e67e22'),
                       ('train/loss_coord','#27ae60'), ('train/loss_eos','#e74c3c')]:
        if col in hist:
            ax.plot(steps, hist[col], alpha=0.2, color=color)
            ax.plot(steps, smooth(hist[col]), color=color, lw=2,
                    label=col.replace('train/',''))
    ax.set_title('Loss components'); ax.legend(fontsize=7); ax.set_xlabel('step')
    ax.set_ylim(0, 6); ax.grid(True, alpha=0.3)

    # face_type_acc
    ax = axes[1]
    if 'train/face_type_acc' in hist:
        ax.plot(steps, hist['train/face_type_acc'], alpha=0.2, color='#8e44ad')
        ax.plot(steps, smooth(hist['train/face_type_acc']), color='#8e44ad', lw=2)
        ax.axhline(0.9, color='gray', ls='--', lw=1, alpha=0.5, label='target 0.90')
    ax.set_title('face_type_acc'); ax.set_ylim(-0.05,1.05); ax.legend()
    ax.set_xlabel('step'); ax.grid(True, alpha=0.3)

    # coord_acc + eos_acc
    ax = axes[2]
    for col, color, label in [
        ('train/coord_acc', '#27ae60', 'coord_acc'),
        ('train/eos_acc',   '#e74c3c', 'eos_acc'),
    ]:
        if col in hist:
            ax.plot(steps, hist[col], alpha=0.2, color=color)
            ax.plot(steps, smooth(hist[col]), color=color, lw=2, label=label)
    ax.set_title('coord_acc + eos_acc'); ax.legend(); ax.set_xlabel('step')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/sanity_check.png', dpi=150)
    plt.show()
    print('Plot saved to /content/sanity_check.png')

## Cell 9 — Resume after disconnect

If the Colab session disconnects:
1. Reconnect and rerun Cells 1–5 (fast: Drive is already mounted, deps cached).
2. In Cell 6, set `RESUME_FROM_CKPT` to the latest checkpoint on Drive, e.g.:
   ```python
   RESUME_FROM_CKPT = '/content/drive/MyDrive/mesh_genai/runs/run_20260613_210000/last.ckpt'
   ```
3. Rerun Cells 6 and 7.

In [ ]:
# List available checkpoints on Drive
import glob as _glob
drive_runs = '/content/drive/MyDrive/mesh_genai/runs'
ckpts = sorted(_glob.glob(os.path.join(drive_runs, '**', '*.ckpt'), recursive=True))
if ckpts:
    print(f'{len(ckpts)} checkpoint(s) on Drive:')
    for p in ckpts[-5:]:   # show last 5
        size_mb = os.path.getsize(p) / 1e6
        print(f'  {p}  ({size_mb:.0f} MB)')
else:
    print('No checkpoints on Drive yet — training has not started or not synced yet.')

## Cell 10 — Scale-up to N=5000

Once the 500-step sanity check passes (see expected values in Cell 8), run this cell to  
download the full 5 000-mesh dataset and restart training from the best checkpoint.

In [ ]:
SCALEUP_TRAIN_DIR = '/content/data/train_5k'
os.makedirs(SCALEUP_TRAIN_DIR, exist_ok=True)

# Download additional 4500 objects (total 5000)
!python scripts/prep_objaverse.py \
    --n 5000 \
    --out_dir {SCALEUP_TRAIN_DIR} \
    --seed 1234 \
    --processes 4

print('Scale-up dataset ready.')
print('Now update TRAIN_DIR in Cell 6 to SCALEUP_TRAIN_DIR and rerun Cells 6-7.')